### Downloading PDFs

In [225]:
from selenium.webdriver.chrome.options import Options
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
import time
import os
import requests
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

BASE_DIR = os.path.join("/Users/aryankhurana/Electricity-Forecasting-Indian-States")
DOWNLOAD_DIR = os.path.join(BASE_DIR, "artifacts", "downloaded_pdfs")

URL = "https://grid-india.in/en/reports/weekly-report"

# Ensure download directory exists
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# === Setup Selenium with Brave ===
options = Options()
options.binary_location = "C:\\Program Files\\BraveSoftware\\Brave-Browser\\Application\\brave.exe"
service = Service(executable_path=r"/Users/aryankhurana/Downloads/Softwares/chromedriver-mac-arm64/chromedriver")
driver = webdriver.Chrome(service=service, options=options)

driver.get(URL)
wait = WebDriverWait(driver, 10)

# === STEP 1: Select "ALL" from dropdown ===
try:
    dropdown_control = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "div.my-select__control")
    ))
    dropdown_control.click()
    time.sleep(1)

    # Wait for dropdown menu to appear
    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div[class*='my-select__menu']")
    ))

    # Find and click ALL option
    all_option = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//div[contains(@class, 'my-select__option') and text()='ALL']")
    ))
    all_option.click()
    time.sleep(2)

    print("Selected ALL option")
    
except Exception as e:
    print("Dropdown selection failed:", e)
    driver.quit()
    exit()

# === STEP 2: Scrape PDF links with pagination ===
all_pdf_links = set()
page_number = 1

while True:
    print(f"Scraping Page {page_number}...")
    time.sleep(2)

    try:
        # Wait for PDF links to load
        wait.until(EC.presence_of_all_elements_located((By.XPATH, "//a[contains(@href, '.pdf')]")))

        pdf_links = driver.find_elements(By.XPATH, "//a[contains(@href, '.pdf')]")
        for link in pdf_links:
            driver.execute_script("window.scrollBy(0, 52);")
            time.sleep(1)

            href = link.get_attribute("href")
            if href and href.endswith(".pdf"):
                all_pdf_links.add(href)

        try:
            next_button = driver.find_element(By.XPATH, "//button[@aria-label='Next Page']")
            if next_button.get_attribute("disabled") is not None:
                print("Reached the last page.")
                break
            next_button.click()
            time.sleep(2)
            page_number += 1
        except Exception as e:
            print("Error navigating to next page:", e)
            break
        # Scroll to the very top of the page
        driver.execute_script("window.scrollTo(0, 100);")
        time.sleep(1)

    except Exception as e:
        print("Error during pagination:", e)
        break
driver.quit()
try:
    print(f"Total PDFs found: {len(all_pdf_links)}. Downloading...")
    for i, link in enumerate(all_pdf_links, 1):
        filename = os.path.join(DOWNLOAD_DIR, f"report_{i:03d}.pdf")
        try:
            response = requests.get(link)
            with open(filename, "wb") as f:
                f.write(response.content)
            print(f"Downloaded: {filename}")
        except Exception as e:
            print(f"Failed to download {link}: {e}")

    print("✅ All downloads complete.")
except Exception as e:
    print(f"Error during download: {e}")



SessionNotCreatedException: Message: session not created
from unknown error: no chrome binary at C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
0   chromedriver                        0x0000000105017570 cxxbridge1$str$ptr + 2731064
1   chromedriver                        0x000000010500f468 cxxbridge1$str$ptr + 2698032
2   chromedriver                        0x0000000104b5e3f8 cxxbridge1$string$len + 90664
3   chromedriver                        0x0000000104b8f6c4 cxxbridge1$string$len + 292084
4   chromedriver                        0x0000000104b8e7d8 cxxbridge1$string$len + 288264
5   chromedriver                        0x0000000104bd2d3c cxxbridge1$string$len + 568172
6   chromedriver                        0x0000000104bd25f0 cxxbridge1$string$len + 566304
7   chromedriver                        0x0000000104

### Extracting Tables from PDFs

In [ ]:
import os

In [ ]:
BASE_DIR = os.path.join("/Users/aryankhurana/Electricity-Forecasting-Indian-States")
DOWNLOAD_DIR = os.path.join(BASE_DIR, "artifacts", "downloaded_pdfs")

In [ ]:
import tabula
import PyPDF2
import pandas as pd

# Folder paths
pdf_dir = os.path.join(BASE_DIR, "artifacts", "downloaded_pdfs")
csv_dir = os.path.join(BASE_DIR, "artifacts", "downloaded_csvs")

os.makedirs(csv_dir, exist_ok=True)

# List all PDF files in the directory
pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith(".pdf")]

# Process each PDF
for idx, file_path in enumerate(pdf_files, 1):
    try:
        print(f"Processing {file_path}...")

        with open(file_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            page_found = False

            for i in range(1, len(reader.pages)):
                text = reader.pages[i].extract_text()
                if text and 'Energy Consumption in States (MUs)' in text:
                    page_found = True
                    required_page = i + 1  # tabula is 1-based
                    break

        if not page_found:
            print(f"❌ Table not found in {file_path}")
            continue

        # Extract table from the page using tabula
        tables = tabula.read_pdf(file_path, pages=required_page, multiple_tables=True)

        if tables:
            df = tables[0]
            # Drop the last row if it is total or NaN
            df = df.dropna(how='all')
            if 'ALL INDIA TOTAL' in df.iloc[-1].astype(str).values:
                df = df.iloc[:-1]

            csv_filename = os.path.splitext(os.path.basename(file_path))[0] + ".csv"
            csv_path = os.path.join(csv_dir, csv_filename)
            df.to_csv(csv_path, index=False)
            print(f"✅ Saved to: {csv_path}")
        else:
            print(f"⚠️ No tables extracted from page {required_page} in {file_path}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")


Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_253.pdf...
✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_253.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_247.pdf...
✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_247.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_290.pdf...
❌ Error processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_290.pdf: EOF marker not found
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_284.pdf...
✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_284.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifa

Got stderr: Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:58:30 AM org.apache.pdfbox.pdmodel.f

✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_279.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_241.pdf...
✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_241.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_255.pdf...
✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_255.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_269.pdf...
❌ Table not found in /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_269.pdf
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_282.pdf...
✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs

Got stderr: Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.font.PDType0Font toUnicode
Jul 24, 2025 12:59:28 AM org.apache.pdfbox.pdmodel.f

✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_143.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_157.pdf...
✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_157.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_180.pdf...
✅ Saved to: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_180.csv
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_194.pdf...
❌ Error processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_194.pdf: EOF marker not found
Processing /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_pdfs/report_427.pdf...
❌ Error processing /Users/aryankhurana/Electricity-Forecasting-Indian-State

# Cleaning

In [ ]:
import os


In [ ]:

directory = "/Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs"


In [ ]:
os.listdir(directory)

['report_266.csv',
 'report_272.csv',
 'report_058.csv',
 'report_064.csv',
 'report_070.csv',
 'report_138.csv',
 'report_110.csv',
 'report_104.csv',
 'report_448.csv',
 'report_312.csv',
 'report_306.csv',
 'report_307.csv',
 'report_313.csv',
 'report_449.csv',
 'report_105.csv',
 'report_111.csv',
 'report_139.csv',
 'report_071.csv',
 'report_065.csv',
 'report_298.csv',
 'report_273.csv',
 'report_267.csv',
 'report_259.csv',
 'report_271.csv',
 'report_265.csv',
 'report_107.csv',
 'report_113.csv',
 'report_339.csv',
 'report_305.csv',
 'report_311.csv',
 'report_310.csv',
 'report_304.csv',
 'report_338.csv',
 'report_112.csv',
 'report_106.csv',
 'report_099.csv',
 'report_066.csv',
 'report_072.csv',
 'report_264.csv',
 'report_270.csv',
 'report_258.csv',
 'report_274.csv',
 'report_260.csv',
 'report_248.csv',
 'report_076.csv',
 'report_062.csv',
 'report_089.csv',
 'report_102.csv',
 'report_116.csv',
 'report_300.csv',
 'report_314.csv',
 'report_328.csv',
 'report_329

In [ ]:
csv_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
csv_files 
len(csv_files)  

391

In [ ]:
def clean_csv_inplace(file_path):

    # Read all lines
    with open(file_path, 'r') as f:
        lines = f.readlines()
        
    if len(lines) <= 1:
        return
        
    original_count = len(lines)
    start_index = 0
    end_index = len(lines)

    first_line = lines[0].strip()
    has_title_problem = False
    if first_line and (
        (first_line[0].isdigit() and '.' in first_line[:5]) or  
        first_line.startswith('*') or
        ('Unnamed:' in first_line and first_line.count('Unnamed:') > 3)
    ):
        has_title_problem = True
        start_index = 1
        
    # Check if last line looks like a note
    last_line = lines[-1].strip()
    has_note_problem = False
    if last_line and (
        last_line.startswith('*') or  # "*Railways_ER ISTS added..."
        last_line.count(',') > 5 or  # mostly commas
        'w.e.f' in last_line.lower() or
        'added as an entity' in last_line.lower()
    ):
        has_note_problem = True
        end_index = len(lines) - 1

    if (has_title_problem or has_note_problem) and (end_index - start_index) > 0:
        print(f"Cleaning {file_path}:")
        cleaned_lines = lines[start_index:end_index]
        # Write back to same file
        with open(file_path, 'w') as f:
            f.writelines(cleaned_lines)

clean_csv_inplace('/Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_366.csv')

Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_366.csv:


In [ ]:
# Loop through all CSV files and apply cleaning only where needed
for csv_path in csv_files:
    clean_csv_inplace(csv_path)
    



Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_266.csv:
Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_272.csv:
Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_058.csv:
Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_064.csv:
Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_070.csv:
Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_138.csv:
Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_110.csv:
Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_104.csv:
Cleaning /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_448.csv:
Cleaning /Users/ary

In [ ]:
def fix_state_formatting(file_path):
    try:
        with open(file_path, 'r') as f:
            lines = f.readlines()
                
        fixed_lines = []
        changes_made = False
        
        for i, line in enumerate(lines):
            # Skip header line
            if i == 0:
                fixed_lines.append(line)
                continue
            
            # Check if line has trailing comma and no leading comma
            stripped_line = line.strip()
            if stripped_line.endswith(',') and not stripped_line.startswith(','):
                # Split the line by commas
                parts = line.rstrip('\n').split(',')
                
                # Check if first part looks like a state name (not a region code)
                first_part = parts[0].strip()
                region_codes = ['NR', 'WR', 'SR', 'ER', 'NER']
                
                # If first part is not a region code and has trailing comma, it's likely a misplaced state
                if first_part not in region_codes and len(first_part) > 2:
                    # Move the state name to the second column
                    new_line = ',' + first_part
                    # Add the rest of the data (skip the trailing empty part from the comma)
                    if len(parts) > 1:
                        new_line += ',' + ','.join(parts[1:-1]) if parts[-1] == '' else ',' + ','.join(parts[1:])
                    new_line += '\n'
                    fixed_lines.append(new_line)
                    changes_made = True
                    continue
            
            # If no issues found, keep the line as is
            fixed_lines.append(line)
        
        # Write back only if changes were made
        if changes_made:
            with open(file_path, 'w') as f:
                f.writelines(fixed_lines)
            print(f"Fixed state formatting: {file_path}")
        else:
            print(f"No formatting issues found: {file_path}")
            
    except Exception as e:
        print(f"Error fixing {file_path}: {e}")
fix_state_formatting('/Users/aryankhurana/Electricity-Forecasting-Indian-States/report_450.csv')


No formatting issues found: /Users/aryankhurana/Electricity-Forecasting-Indian-States/report_450.csv


In [ ]:
for csv_path in csv_files:
    fix_state_formatting(csv_path)
    

Fixed state formatting: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_266.csv
Fixed state formatting: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_272.csv
Fixed state formatting: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_058.csv
Fixed state formatting: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_064.csv
Fixed state formatting: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_070.csv
Fixed state formatting: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_138.csv
Fixed state formatting: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_110.csv
Fixed state formatting: /Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/downloaded_csvs/report_104.csv
Fixed state form

# Merging the csv files into one file

In [ ]:
import pandas as pd
import os

In [ ]:
import pandas as pd
import os

def merge_all_csv_files(csv_files, output_path):
    """
    Merge all CSV files into a single file based on States column
    """
    all_dataframes = []
    
    for i, csv_path in enumerate(csv_files):
        try:
            # Read the CSV file
            df = pd.read_csv(csv_path)
            
            # Drop Region column and any Unnamed columns
            columns_to_drop = []
            for col in df.columns:
                if col == 'Region' or col.startswith('Unnamed'):
                    columns_to_drop.append(col)
            
            if columns_to_drop:
                df = df.drop(columns=columns_to_drop)
                print(f"Dropped columns {columns_to_drop} from {os.path.basename(csv_path)}")
            
            # Check if States column exists
            if 'States' not in df.columns:
                print(f"Warning: No 'States' column found in {os.path.basename(csv_path)}, skipping...")
                continue
            
            # Remove rows where States column is empty or NaN
            df = df.dropna(subset=['States'])
            df = df[df['States'].str.strip() != '']
            
            if df.empty:
                print(f"Warning: No valid data in {os.path.basename(csv_path)}, skipping...")
                continue
            
            all_dataframes.append(df)
            print(f"Processed: {os.path.basename(csv_path)} - {len(df)} rows")
            
        except Exception as e:
            print(f"Error processing {csv_path}: {e}")
            continue
        
    # Start with the first dataframe
    merged_df = all_dataframes[0]
    
    # Merge each subsequent dataframe
    for i in range(1, len(all_dataframes)):
        merged_df = pd.merge(merged_df, all_dataframes[i], on='States', how='outer')   
    merged_df.to_csv(output_path, index=False)
    
    return merged_df

# Usage
output_file = "/Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/merged_electricity_data.csv"
merged_data = merge_all_csv_files(csv_files, output_file)



Dropped columns ['Region'] from report_266.csv
Processed: report_266.csv - 35 rows
Dropped columns ['Region'] from report_272.csv
Processed: report_272.csv - 35 rows
Dropped columns ['Region'] from report_058.csv
Processed: report_058.csv - 35 rows
Dropped columns ['Region'] from report_064.csv
Processed: report_064.csv - 39 rows
Dropped columns ['Region'] from report_070.csv
Processed: report_070.csv - 35 rows
Dropped columns ['Region'] from report_138.csv
Processed: report_138.csv - 35 rows
Dropped columns ['Region'] from report_110.csv
Processed: report_110.csv - 35 rows
Dropped columns ['Region'] from report_104.csv
Processed: report_104.csv - 35 rows
Dropped columns ['Region'] from report_448.csv
Processed: report_448.csv - 35 rows
Dropped columns ['Region'] from report_312.csv
Processed: report_312.csv - 39 rows
Dropped columns ['Region'] from report_306.csv
Processed: report_306.csv - 35 rows
Dropped columns ['Region'] from report_307.csv
Processed: report_307.csv - 35 rows
Drop

# Cleaning the data

In [262]:
df = pd.read_csv('/Users/aryankhurana/Electricity-Forecasting-Indian-States/artifacts/merged_electricity_data.csv')
df.head(50)

,States,04-04-2022,05-04-2022,06-04-2022,07-04-2022,08-04-2022,09-04-2022,10-04-2022,08-01-2018,09-01-2018,...,15-09-2023,16-09-2023,17-09-2023,11-04-2022,12-04-2022,13-04-2022,14-04-2022,15-04-2022,16-04-2022,17-04-2022
0,ALL INDIA TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4795.3,4654.1,4284.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AMNSIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,18.5,18.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andhra Pradesh,220.8,218.7,205.9,213.1,217.1,211.8,207.4,162.2,165.2,...,230.9,230.3,226.5,209.4,208.8,209.2,213.0,211.6,210.5,200.7
3,Arunachal Pradesh,2.2,2.3,2.4,2.4,2.4,2.3,2.3,2.3,2.2,...,3.0,2.9,3.1,2.4,2.4,2.2,2.2,1.5,2.0,2.1
4,Assam,23.8,26.3,26.8,27.8,27.9,26.1,22.9,22.8,23.6,...,50.0,49.1,47.1,23.2,27.8,28.6,22.1,19.8,20.7,21.2
5,BALCO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.4,12.4,12.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Bihar,114.4,110.1,112.7,119.2,118.9,116.7,110.0,71.5,73.9,...,147.9,152.5,152.9,114.1,117.3,121.0,122.4,118.8,123.6,125.2
7,Chandigarh,4.5,4.6,4.6,4.6,4.7,4.6,4.3,3.7,3.8,...,6.6,6.2,6.0,5.0,5.3,5.5,4.7,4.6,4.6,4.3
8,Chhattisgarh,123.3,125.8,123.4,122.7,124.8,122.2,120.2,73.3,73.4,...,92.8,96.5,93.7,120.6,122.6,120.5,123.9,123.7,125.0,123.4
9,DD,7.5,7.9,8.0,8.0,8.2,8.2,7.4,7.0,7.2,...,NaN,NaN,NaN,7.7,8.0,8.1,8.0,8.1,8.0,7.3


In [263]:
df.drop(0,axis=0, inplace=True)  # Drop the first row if it is not needed

In [264]:
df.head(50)

,States,04-04-2022,05-04-2022,06-04-2022,07-04-2022,08-04-2022,09-04-2022,10-04-2022,08-01-2018,09-01-2018,...,15-09-2023,16-09-2023,17-09-2023,11-04-2022,12-04-2022,13-04-2022,14-04-2022,15-04-2022,16-04-2022,17-04-2022
1,AMNSIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,18.5,18.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andhra Pradesh,220.8,218.7,205.9,213.1,217.1,211.8,207.4,162.2,165.2,...,230.9,230.3,226.5,209.4,208.8,209.2,213.0,211.6,210.5,200.7
3,Arunachal Pradesh,2.2,2.3,2.4,2.4,2.4,2.3,2.3,2.3,2.2,...,3.0,2.9,3.1,2.4,2.4,2.2,2.2,1.5,2.0,2.1
4,Assam,23.8,26.3,26.8,27.8,27.9,26.1,22.9,22.8,23.6,...,50.0,49.1,47.1,23.2,27.8,28.6,22.1,19.8,20.7,21.2
5,BALCO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.4,12.4,12.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Bihar,114.4,110.1,112.7,119.2,118.9,116.7,110.0,71.5,73.9,...,147.9,152.5,152.9,114.1,117.3,121.0,122.4,118.8,123.6,125.2
7,Chandigarh,4.5,4.6,4.6,4.6,4.7,4.6,4.3,3.7,3.8,...,6.6,6.2,6.0,5.0,5.3,5.5,4.7,4.6,4.6,4.3
8,Chhattisgarh,123.3,125.8,123.4,122.7,124.8,122.2,120.2,73.3,73.4,...,92.8,96.5,93.7,120.6,122.6,120.5,123.9,123.7,125.0,123.4
9,DD,7.5,7.9,8.0,8.0,8.2,8.2,7.4,7.0,7.2,...,NaN,NaN,NaN,7.7,8.0,8.1,8.0,8.1,8.0,7.3
10,DD*,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [265]:
import numpy as np

In [266]:
def fill_from_starred_rows(df, base_states=['DD', 'DNH','Railways_ER ISTS']):
    """
    Fill NaN values in base state rows with values from their starred counterparts
    
    Parameters:
    df: DataFrame with electricity data
    base_states: List of state names to process (without asterisk)
    """
    df_result = df.copy()
    
    for state in base_states:
        base_mask = df_result['States'] == state
        star_mask = df_result['States'] == f'{state}*'
        
        if base_mask.any() and star_mask.any():
            base_idx = df_result[base_mask].index[0]
            star_idx = df_result[star_mask].index[0]
            
            # Get numeric columns only
            numeric_cols = df_result.select_dtypes(include=[np.number]).columns
            
            # Fill NaN values in base row with values from starred row
            for col in numeric_cols:
                if pd.isna(df_result.loc[base_idx, col]) and not pd.isna(df_result.loc[star_idx, col]):
                    df_result.loc[base_idx, col] = df_result.loc[star_idx, col]
            
            # Remove the starred row after filling
            df_result = df_result.drop(star_idx)
    
    return df_result.reset_index(drop=True)

# Usage
df = fill_from_starred_rows(df, ['DD', 'DNH'])

In [267]:
df.head(50)

,States,04-04-2022,05-04-2022,06-04-2022,07-04-2022,08-04-2022,09-04-2022,10-04-2022,08-01-2018,09-01-2018,...,15-09-2023,16-09-2023,17-09-2023,11-04-2022,12-04-2022,13-04-2022,14-04-2022,15-04-2022,16-04-2022,17-04-2022
0,AMNSIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,18.5,18.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Andhra Pradesh,220.8,218.7,205.9,213.1,217.1,211.8,207.4,162.2,165.2,...,230.9,230.3,226.5,209.4,208.8,209.2,213.0,211.6,210.5,200.7
2,Arunachal Pradesh,2.2,2.3,2.4,2.4,2.4,2.3,2.3,2.3,2.2,...,3.0,2.9,3.1,2.4,2.4,2.2,2.2,1.5,2.0,2.1
3,Assam,23.8,26.3,26.8,27.8,27.9,26.1,22.9,22.8,23.6,...,50.0,49.1,47.1,23.2,27.8,28.6,22.1,19.8,20.7,21.2
4,BALCO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.4,12.4,12.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Bihar,114.4,110.1,112.7,119.2,118.9,116.7,110.0,71.5,73.9,...,147.9,152.5,152.9,114.1,117.3,121.0,122.4,118.8,123.6,125.2
6,Chandigarh,4.5,4.6,4.6,4.6,4.7,4.6,4.3,3.7,3.8,...,6.6,6.2,6.0,5.0,5.3,5.5,4.7,4.6,4.6,4.3
7,Chhattisgarh,123.3,125.8,123.4,122.7,124.8,122.2,120.2,73.3,73.4,...,92.8,96.5,93.7,120.6,122.6,120.5,123.9,123.7,125.0,123.4
8,DD,7.5,7.9,8.0,8.0,8.2,8.2,7.4,7.0,7.2,...,NaN,NaN,NaN,7.7,8.0,8.1,8.0,8.1,8.0,7.3
9,DNH,18.4,19.6,20.2,19.2,20.3,20.5,20.1,17.3,17.6,...,NaN,NaN,NaN,20.5,20.5,20.1,20.4,20.2,20.1,19.9


In [268]:
def add_total_consumption_row(df):
    """
    Add a 'Total Consumption' row that sums all electricity consumption
    """
    df_result = df.copy()
    
    # Get all numeric columns (these should be the date columns with consumption data)
    numeric_cols = df_result.select_dtypes(include=[np.number]).columns
    
    # Calculate the sum for each numeric column, ignoring NaN values
    total_row = {}
    total_row['States'] = 'Total Consumption'
    
    for col in numeric_cols:
        total_row[col] = df_result[col].sum(skipna=True)
    
    # Convert to DataFrame row and append
    total_df = pd.DataFrame([total_row])
    df_result = pd.concat([df_result, total_df], ignore_index=True)
    
    return df_result

# Apply the function
df = add_total_consumption_row(df)
df.head(50)

,States,04-04-2022,05-04-2022,06-04-2022,07-04-2022,08-04-2022,09-04-2022,10-04-2022,08-01-2018,09-01-2018,...,15-09-2023,16-09-2023,17-09-2023,11-04-2022,12-04-2022,13-04-2022,14-04-2022,15-04-2022,16-04-2022,17-04-2022
0,AMNSIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,18.0,18.5,18.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Andhra Pradesh,220.8,218.7,205.9,213.1,217.1,211.8,207.4,162.2,165.2,...,230.9,230.3,226.5,209.4,208.8,209.2,213.0,211.6,210.5,200.7
2,Arunachal Pradesh,2.2,2.3,2.4,2.4,2.4,2.3,2.3,2.3,2.2,...,3.0,2.9,3.1,2.4,2.4,2.2,2.2,1.5,2.0,2.1
3,Assam,23.8,26.3,26.8,27.8,27.9,26.1,22.9,22.8,23.6,...,50.0,49.1,47.1,23.2,27.8,28.6,22.1,19.8,20.7,21.2
4,BALCO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.4,12.4,12.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Bihar,114.4,110.1,112.7,119.2,118.9,116.7,110.0,71.5,73.9,...,147.9,152.5,152.9,114.1,117.3,121.0,122.4,118.8,123.6,125.2
6,Chandigarh,4.5,4.6,4.6,4.6,4.7,4.6,4.3,3.7,3.8,...,6.6,6.2,6.0,5.0,5.3,5.5,4.7,4.6,4.6,4.3
7,Chhattisgarh,123.3,125.8,123.4,122.7,124.8,122.2,120.2,73.3,73.4,...,92.8,96.5,93.7,120.6,122.6,120.5,123.9,123.7,125.0,123.4
8,DD,7.5,7.9,8.0,8.0,8.2,8.2,7.4,7.0,7.2,...,NaN,NaN,NaN,7.7,8.0,8.1,8.0,8.1,8.0,7.3
9,DNH,18.4,19.6,20.2,19.2,20.3,20.5,20.1,17.3,17.6,...,NaN,NaN,NaN,20.5,20.5,20.1,20.4,20.2,20.1,19.9


In [269]:
def count_nan_per_row(df):
    """
    Count NaN values for each row and add as a new column
    """
    df_result = df.copy()
    
    # Count NaN values across all columns for each row
    df_result['NaN_Count'] = df_result.isnull().sum(axis=1)
    
    return df_result

# Apply the function
x = count_nan_per_row(df)

# Display the States column and NaN count
print(x[['States', 'NaN_Count']])

               States  NaN_Count
0              AMNSIL       2163
1      Andhra Pradesh          0
2   Arunachal Pradesh          0
3               Assam          0
4               BALCO       2163
5               Bihar          0
6          Chandigarh          0
7        Chhattisgarh          0
8                  DD        707
9                 DNH        712
10          DNHDDPDCL       1988
11                DVC          0
12              Delhi          0
13        Essar steel        532
14                Goa          0
15            Gujarat          0
16                 HP          0
17            Haryana          0
18                J&K          0
19          Jharkhand          0
20          Karnataka          0
21             Kerala          0
22                 MP          0
23        Maharashtra          0
24            Manipur          0
25          Meghalaya         14
26            Mizoram          0
27      NER Meghalaya       2681
28           Nagaland          0
29        

In [270]:
# Apply the function
x = count_nan_per_row(df)

# Display the States column and NaN count
print(x[['States', 'NaN_Count']])

               States  NaN_Count
0              AMNSIL       2163
1      Andhra Pradesh          0
2   Arunachal Pradesh          0
3               Assam          0
4               BALCO       2163
5               Bihar          0
6          Chandigarh          0
7        Chhattisgarh          0
8                  DD        707
9                 DNH        712
10          DNHDDPDCL       1988
11                DVC          0
12              Delhi          0
13        Essar steel        532
14                Goa          0
15            Gujarat          0
16                 HP          0
17            Haryana          0
18                J&K          0
19          Jharkhand          0
20          Karnataka          0
21             Kerala          0
22                 MP          0
23        Maharashtra          0
24            Manipur          0
25          Meghalaya         14
26            Mizoram          0
27      NER Meghalaya       2681
28           Nagaland          0
29        

In [271]:
df.drop([0,4,10,27,30,32,33,34],axis=0, inplace=True)

In [272]:
df.set_index('States', inplace=True)

In [273]:
df.head(50)

,04-04-2022,05-04-2022,06-04-2022,07-04-2022,08-04-2022,09-04-2022,10-04-2022,08-01-2018,09-01-2018,10-01-2018,...,15-09-2023,16-09-2023,17-09-2023,11-04-2022,12-04-2022,13-04-2022,14-04-2022,15-04-2022,16-04-2022,17-04-2022
States,,,,,,,,,,,,,,,,,,,,,
Andhra Pradesh,220.8,218.7,205.9,213.1,217.1,211.8,207.4,162.2,165.2,165.6,...,230.9,230.3,226.5,209.4,208.8,209.2,213.0,211.6,210.5,200.7
Arunachal Pradesh,2.2,2.3,2.4,2.4,2.4,2.3,2.3,2.3,2.2,2.2,...,3.0,2.9,3.1,2.4,2.4,2.2,2.2,1.5,2.0,2.1
Assam,23.8,26.3,26.8,27.8,27.9,26.1,22.9,22.8,23.6,23.8,...,50.0,49.1,47.1,23.2,27.8,28.6,22.1,19.8,20.7,21.2
Bihar,114.4,110.1,112.7,119.2,118.9,116.7,110.0,71.5,73.9,74.7,...,147.9,152.5,152.9,114.1,117.3,121.0,122.4,118.8,123.6,125.2
Chandigarh,4.5,4.6,4.6,4.6,4.7,4.6,4.3,3.7,3.8,3.8,...,6.6,6.2,6.0,5.0,5.3,5.5,4.7,4.6,4.6,4.3
Chhattisgarh,123.3,125.8,123.4,122.7,124.8,122.2,120.2,73.3,73.4,73.6,...,92.8,96.5,93.7,120.6,122.6,120.5,123.9,123.7,125.0,123.4
DD,7.5,7.9,8.0,8.0,8.2,8.2,7.4,7.0,7.2,7.3,...,NaN,NaN,NaN,7.7,8.0,8.1,8.0,8.1,8.0,7.3
DNH,18.4,19.6,20.2,19.2,20.3,20.5,20.1,17.3,17.6,17.4,...,NaN,NaN,NaN,20.5,20.5,20.1,20.4,20.2,20.1,19.9
DVC,77.4,79.5,80.3,76.7,80.3,76.8,75.4,69.2,70.8,69.0,...,77.4,79.2,75.4,74.9,75.0,78.8,79.7,78.5,79.4,78.5


In [274]:
df = df.T
df.head(40)

States,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,DNH,DVC,Delhi,...,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption
04-04-2022,220.8,2.2,23.8,114.4,4.5,123.3,7.5,18.4,77.4,94.5,...,149.5,249.4,1.7,363.1,265.7,NaN,397.5,39.8,178.1,4393.7
05-04-2022,218.7,2.3,26.3,110.1,4.6,125.8,7.9,19.6,79.5,95.7,...,142.2,253.7,1.8,367.1,266.0,NaN,395.7,40.6,185.0,4431.5
06-04-2022,205.9,2.4,26.8,112.7,4.6,123.4,8.0,20.2,80.3,97.9,...,147.1,254.3,1.8,367.6,265.7,NaN,405.1,37.5,185.3,4423.3
07-04-2022,213.1,2.4,27.8,119.2,4.6,122.7,8.0,19.2,76.7,99.4,...,161.3,249.6,1.6,372.7,267.1,NaN,407.8,38.8,188.2,4466.5
08-04-2022,217.1,2.4,27.9,118.9,4.7,124.8,8.2,20.3,80.3,101.9,...,160.0,260.1,1.3,372.7,265.0,NaN,412.5,40.9,186.6,4515.1
09-04-2022,211.8,2.3,26.1,116.7,4.6,122.2,8.2,20.5,76.8,100.7,...,158.9,262.9,1.0,360.1,253.0,NaN,416.9,41.6,186.9,4471.3
10-04-2022,207.4,2.3,22.9,110.0,4.3,120.2,7.4,20.1,75.4,98.5,...,147.2,256.4,1.0,320.3,254.4,NaN,407.2,39.0,178.2,4296.2
08-01-2018,162.2,2.3,22.8,71.5,3.7,73.3,7.0,17.3,69.2,68.6,...,105.4,209.3,1.6,288.9,177.8,NaN,305.3,38.3,103.3,3248.3
09-01-2018,165.2,2.2,23.6,73.9,3.8,73.4,7.2,17.6,70.8,68.9,...,108.2,211.5,1.6,292.5,179.7,NaN,310.5,39.2,111.5,3300.3
10-01-2018,165.6,2.2,23.8,74.7,3.8,73.6,7.3,17.4,69.0,69.3,...,108.7,216.9,1.8,279.8,180.7,NaN,309.9,39.9,113.2,3308.7


In [276]:
df.to_csv('/Users/aryankhurana/Electricity-Forecasting-Indian-States/clean_data.csv')

In [277]:
df = pd.read_csv('/Users/aryankhurana/Electricity-Forecasting-Indian-States/clean_data.csv')
df.head(50)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,DNH,DVC,...,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption
0,04-04-2022,220.8,2.2,23.8,114.4,4.5,123.3,7.5,18.4,77.4,...,149.5,249.4,1.7,363.1,265.7,NaN,397.5,39.8,178.1,4393.7
1,05-04-2022,218.7,2.3,26.3,110.1,4.6,125.8,7.9,19.6,79.5,...,142.2,253.7,1.8,367.1,266.0,NaN,395.7,40.6,185.0,4431.5
2,06-04-2022,205.9,2.4,26.8,112.7,4.6,123.4,8.0,20.2,80.3,...,147.1,254.3,1.8,367.6,265.7,NaN,405.1,37.5,185.3,4423.3
3,07-04-2022,213.1,2.4,27.8,119.2,4.6,122.7,8.0,19.2,76.7,...,161.3,249.6,1.6,372.7,267.1,NaN,407.8,38.8,188.2,4466.5
4,08-04-2022,217.1,2.4,27.9,118.9,4.7,124.8,8.2,20.3,80.3,...,160.0,260.1,1.3,372.7,265.0,NaN,412.5,40.9,186.6,4515.1
5,09-04-2022,211.8,2.3,26.1,116.7,4.6,122.2,8.2,20.5,76.8,...,158.9,262.9,1.0,360.1,253.0,NaN,416.9,41.6,186.9,4471.3
6,10-04-2022,207.4,2.3,22.9,110.0,4.3,120.2,7.4,20.1,75.4,...,147.2,256.4,1.0,320.3,254.4,NaN,407.2,39.0,178.2,4296.2
7,08-01-2018,162.2,2.3,22.8,71.5,3.7,73.3,7.0,17.3,69.2,...,105.4,209.3,1.6,288.9,177.8,NaN,305.3,38.3,103.3,3248.3
8,09-01-2018,165.2,2.2,23.6,73.9,3.8,73.4,7.2,17.6,70.8,...,108.2,211.5,1.6,292.5,179.7,NaN,310.5,39.2,111.5,3300.3
9,10-01-2018,165.6,2.2,23.8,74.7,3.8,73.6,7.3,17.4,69.0,...,108.7,216.9,1.8,279.8,180.7,NaN,309.9,39.9,113.2,3308.7


In [278]:
from datetime import datetime

def convert_date_format(df):
    """
    Convert date format from dd-mm-yyyy to yyyy-mm-dd
    """
    df_result = df.copy()
    
    # Function to parse and convert date format
    def parse_date(date_str):
        try:
            # Parse the date string (assuming it's in dd-mm-yyyy format)
            # From your screenshot, it looks like: 04-04-2022, 05-04-2022, etc.
            date_obj = pd.to_datetime(date_str, format='%d-%m-%Y')
            # Convert to yyyy-mm-dd format
            return date_obj.strftime('%Y-%m-%d')
        except:
            # If parsing fails, return original value
            return date_str
    
    # Apply the conversion to the dates column
    df_result['Dates'] = df_result['Dates'].apply(parse_date)
    
    return df_result

# Apply the function
df = convert_date_format(df)
df.head(50)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,DNH,DVC,...,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption
0,2022-04-04,220.8,2.2,23.8,114.4,4.5,123.3,7.5,18.4,77.4,...,149.5,249.4,1.7,363.1,265.7,NaN,397.5,39.8,178.1,4393.7
1,2022-04-05,218.7,2.3,26.3,110.1,4.6,125.8,7.9,19.6,79.5,...,142.2,253.7,1.8,367.1,266.0,NaN,395.7,40.6,185.0,4431.5
2,2022-04-06,205.9,2.4,26.8,112.7,4.6,123.4,8.0,20.2,80.3,...,147.1,254.3,1.8,367.6,265.7,NaN,405.1,37.5,185.3,4423.3
3,2022-04-07,213.1,2.4,27.8,119.2,4.6,122.7,8.0,19.2,76.7,...,161.3,249.6,1.6,372.7,267.1,NaN,407.8,38.8,188.2,4466.5
4,2022-04-08,217.1,2.4,27.9,118.9,4.7,124.8,8.2,20.3,80.3,...,160.0,260.1,1.3,372.7,265.0,NaN,412.5,40.9,186.6,4515.1
5,2022-04-09,211.8,2.3,26.1,116.7,4.6,122.2,8.2,20.5,76.8,...,158.9,262.9,1.0,360.1,253.0,NaN,416.9,41.6,186.9,4471.3
6,2022-04-10,207.4,2.3,22.9,110.0,4.3,120.2,7.4,20.1,75.4,...,147.2,256.4,1.0,320.3,254.4,NaN,407.2,39.0,178.2,4296.2
7,2018-01-08,162.2,2.3,22.8,71.5,3.7,73.3,7.0,17.3,69.2,...,105.4,209.3,1.6,288.9,177.8,NaN,305.3,38.3,103.3,3248.3
8,2018-01-09,165.2,2.2,23.6,73.9,3.8,73.4,7.2,17.6,70.8,...,108.2,211.5,1.6,292.5,179.7,NaN,310.5,39.2,111.5,3300.3
9,2018-01-10,165.6,2.2,23.8,74.7,3.8,73.6,7.3,17.4,69.0,...,108.7,216.9,1.8,279.8,180.7,NaN,309.9,39.9,113.2,3308.7


In [279]:
df.to_csv('/Users/aryankhurana/Electricity-Forecasting-Indian-States/clean_data.csv')

--------------------------------------------------------------------------------------------------------------------------------------------

In [297]:
data = pd.read_csv('/Users/aryankhurana/Electricity-Forecasting-Indian-States/data_clean.csv')
data.head(5)

,Unnamed: 0,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,Delhi,...,Pondy,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal
0,0,2013-01-06,251.5,1.6,21.3,40.9,3.8,60.3,5.8,66.0,...,5.6,95.2,192.0,1.7,254.5,127.6,2.6,225.0,33.6,107.5
1,1,2013-01-07,257.9,1.7,21.2,39.6,3.8,64.3,5.8,67.2,...,5.5,92.8,192.9,1.8,262.3,129.2,2.6,219.1,34.2,106.6
2,2,2013-01-08,265.1,1.6,20.4,39.2,3.9,63.0,5.9,66.7,...,5.8,93.9,197.1,1.8,256.6,131.9,2.1,238.7,33.9,103.9
3,3,2013-01-09,261.0,1.6,21.4,39.5,3.7,65.0,5.8,67.4,...,5.9,90.1,197.6,1.9,254.6,135.0,2.7,229.1,32.7,105.7
4,4,2013-01-10,260.0,1.6,21.1,39.7,3.8,66.5,5.9,67.8,...,5.9,93.5,199.9,1.9,255.3,130.0,3.2,224.5,35.4,105.3


In [298]:
data.drop('Unnamed: 0', axis=1, inplace=True)
data.head(5)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,Delhi,DNH,...,Pondy,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal
0,2013-01-06,251.5,1.6,21.3,40.9,3.8,60.3,5.8,66.0,15.0,...,5.6,95.2,192.0,1.7,254.5,127.6,2.6,225.0,33.6,107.5
1,2013-01-07,257.9,1.7,21.2,39.6,3.8,64.3,5.8,67.2,15.1,...,5.5,92.8,192.9,1.8,262.3,129.2,2.6,219.1,34.2,106.6
2,2013-01-08,265.1,1.6,20.4,39.2,3.9,63.0,5.9,66.7,15.2,...,5.8,93.9,197.1,1.8,256.6,131.9,2.1,238.7,33.9,103.9
3,2013-01-09,261.0,1.6,21.4,39.5,3.7,65.0,5.8,67.4,15.2,...,5.9,90.1,197.6,1.9,254.6,135.0,2.7,229.1,32.7,105.7
4,2013-01-10,260.0,1.6,21.1,39.7,3.8,66.5,5.9,67.8,14.9,...,5.9,93.5,199.9,1.9,255.3,130.0,3.2,224.5,35.4,105.3


In [300]:
data['Total Consumption'] = data.iloc[:, 1:].sum(axis=1)
data.head(5)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,Delhi,DNH,...,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption
0,2013-01-06,251.5,1.6,21.3,40.9,3.8,60.3,5.8,66.0,15.0,...,95.2,192.0,1.7,254.5,127.6,2.6,225.0,33.6,107.5,2845.7
1,2013-01-07,257.9,1.7,21.2,39.6,3.8,64.3,5.8,67.2,15.1,...,92.8,192.9,1.8,262.3,129.2,2.6,219.1,34.2,106.6,2876.7
2,2013-01-08,265.1,1.6,20.4,39.2,3.9,63.0,5.9,66.7,15.2,...,93.9,197.1,1.8,256.6,131.9,2.1,238.7,33.9,103.9,2923.0
3,2013-01-09,261.0,1.6,21.4,39.5,3.7,65.0,5.8,67.4,15.2,...,90.1,197.6,1.9,254.6,135.0,2.7,229.1,32.7,105.7,2912.1
4,2013-01-10,260.0,1.6,21.1,39.7,3.8,66.5,5.9,67.8,14.9,...,93.5,199.9,1.9,255.3,130.0,3.2,224.5,35.4,105.3,2920.8


In [301]:
df.head(5)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,DNH,DVC,...,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption
0,2022-04-04,220.8,2.2,23.8,114.4,4.5,123.3,7.5,18.4,77.4,...,149.5,249.4,1.7,363.1,265.7,NaN,397.5,39.8,178.1,4393.7
1,2022-04-05,218.7,2.3,26.3,110.1,4.6,125.8,7.9,19.6,79.5,...,142.2,253.7,1.8,367.1,266.0,NaN,395.7,40.6,185.0,4431.5
2,2022-04-06,205.9,2.4,26.8,112.7,4.6,123.4,8.0,20.2,80.3,...,147.1,254.3,1.8,367.6,265.7,NaN,405.1,37.5,185.3,4423.3
3,2022-04-07,213.1,2.4,27.8,119.2,4.6,122.7,8.0,19.2,76.7,...,161.3,249.6,1.6,372.7,267.1,NaN,407.8,38.8,188.2,4466.5
4,2022-04-08,217.1,2.4,27.9,118.9,4.7,124.8,8.2,20.3,80.3,...,160.0,260.1,1.3,372.7,265.0,NaN,412.5,40.9,186.6,4515.1


In [304]:
combined_df = pd.concat([data, df], ignore_index=True)


In [305]:
combined_df.head(500)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,Delhi,DNH,...,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption
0,2013-01-06,251.5,1.6,21.3,40.9,3.8,60.3,5.8,66.0,15.0,...,95.2,192.0,1.7,254.5,127.6,2.6,225.0,33.6,107.5,2845.7
1,2013-01-07,257.9,1.7,21.2,39.6,3.8,64.3,5.8,67.2,15.1,...,92.8,192.9,1.8,262.3,129.2,2.6,219.1,34.2,106.6,2876.7
2,2013-01-08,265.1,1.6,20.4,39.2,3.9,63.0,5.9,66.7,15.2,...,93.9,197.1,1.8,256.6,131.9,2.1,238.7,33.9,103.9,2923.0
3,2013-01-09,261.0,1.6,21.4,39.5,3.7,65.0,5.8,67.4,15.2,...,90.1,197.6,1.9,254.6,135.0,2.7,229.1,32.7,105.7,2912.1
4,2013-01-10,260.0,1.6,21.1,39.7,3.8,66.5,5.9,67.8,14.9,...,93.5,199.9,1.9,255.3,130.0,3.2,224.5,35.4,105.3,2920.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2015-05-03,135.4,1.2,19.8,56.6,4.0,76.8,6.2,81.9,16.2,...,117.5,183.7,1.1,258.6,121.6,3.0,248.0,32.4,122.3,2910.8
496,2015-05-04,142.1,1.2,19.3,55.3,4.7,76.1,6.4,86.1,16.4,...,124.1,185.0,1.1,271.4,124.4,2.9,263.4,33.7,124.1,3006.5
497,2015-05-05,142.9,1.1,19.0,55.3,4.9,79.9,6.4,92.5,16.1,...,126.6,186.1,1.4,279.9,119.0,3.2,260.7,34.5,141.9,3031.3
498,2015-05-06,144.4,1.0,21.9,59.1,5.2,79.6,6.2,95.2,16.3,...,129.6,182.8,1.4,282.2,122.3,2.3,269.9,36.2,145.8,3079.8


In [ ]:
combined_df = combined_df.drop_duplicates()

In [307]:
# Sort by Dates column
combined_df = combined_df.sort_values('Dates')

In [308]:
combined_df = combined_df.reset_index(drop=True)
combined_df.head(500)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,Delhi,DNH,...,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption
0,2013-01-06,251.5,1.6,21.3,40.9,3.8,60.3,5.8,66.0,15.0,...,95.2,192.0,1.7,254.5,127.6,2.6,225.0,33.6,107.5,2845.7
1,2013-01-07,257.9,1.7,21.2,39.6,3.8,64.3,5.8,67.2,15.1,...,92.8,192.9,1.8,262.3,129.2,2.6,219.1,34.2,106.6,2876.7
2,2013-01-08,265.1,1.6,20.4,39.2,3.9,63.0,5.9,66.7,15.2,...,93.9,197.1,1.8,256.6,131.9,2.1,238.7,33.9,103.9,2923.0
3,2013-01-09,261.0,1.6,21.4,39.5,3.7,65.0,5.8,67.4,15.2,...,90.1,197.6,1.9,254.6,135.0,2.7,229.1,32.7,105.7,2912.1
4,2013-01-10,260.0,1.6,21.1,39.7,3.8,66.5,5.9,67.8,14.9,...,93.5,199.9,1.9,255.3,130.0,3.2,224.5,35.4,105.3,2920.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2015-05-03,135.4,1.2,19.8,56.6,4.0,76.8,6.2,81.9,16.2,...,117.5,183.7,1.1,258.6,121.6,3.0,248.0,32.4,122.3,2910.8
496,2015-05-04,142.1,1.2,19.3,55.3,4.7,76.1,6.4,86.1,16.4,...,124.1,185.0,1.1,271.4,124.4,2.9,263.4,33.7,124.1,3006.5
497,2015-05-05,142.9,1.1,19.0,55.3,4.9,79.9,6.4,92.5,16.1,...,126.6,186.1,1.4,279.9,119.0,3.2,260.7,34.5,141.9,3031.3
498,2015-05-06,144.4,1.0,21.9,59.1,5.2,79.6,6.2,95.2,16.3,...,129.6,182.8,1.4,282.2,122.3,2.3,269.9,36.2,145.8,3079.8


In [309]:
combined_df.tail(50)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,Delhi,DNH,...,Punjab,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption
4912,2024-08-13,226.2,3.0,45.0,148.7,7.0,121.4,NaN,121.9,NaN,...,293.8,246.7,1.0,349.3,290.6,NaN,509.0,50.1,221.5,4770.5
4913,2024-08-14,228.9,3.0,51.1,145.7,6.9,120.3,NaN,128.1,NaN,...,311.4,248.7,1.2,354.9,294.4,NaN,526.4,50.8,221.9,4867.4
4914,2024-08-15,230.9,2.9,41.0,149.1,5.7,114.2,NaN,113.0,NaN,...,292.5,230.8,0.7,313.4,287.5,NaN,553.9,43.9,209.6,4600.8
4915,2024-08-16,235.4,3.2,44.6,153.8,6.7,111.7,NaN,129.7,NaN,...,307.7,223.5,1.1,345.8,288.5,NaN,556.0,49.1,217.8,4800.6
4916,2024-08-17,227.9,3.0,48.8,162.2,6.6,114.2,NaN,132.6,NaN,...,289.7,236.5,1.2,348.5,270.8,NaN,579.1,53.0,222.5,4856.5
4917,2024-08-18,228.2,2.9,43.1,141.3,6.0,113.1,NaN,130.4,NaN,...,302.5,250.1,0.9,329.2,273.7,NaN,521.2,49.9,197.3,4691.7
4918,2024-08-19,231.2,2.9,42.0,150.3,6.6,113.4,NaN,129.0,NaN,...,292.7,249.1,1.3,350.7,263.2,NaN,532.4,40.3,202.1,4677.5
4919,2024-08-20,225.6,3.0,36.5,146.2,7.0,116.5,NaN,126.3,NaN,...,284.5,267.4,1.4,353.5,231.2,NaN,502.2,45.5,206.7,4673.4
4920,2024-08-21,227.2,3.0,40.7,150.9,7.4,120.4,NaN,129.7,NaN,...,295.0,286.8,1.3,350.2,256.6,NaN,520.5,47.3,215.3,4827.9
4921,2024-08-22,218.0,3.0,46.1,149.4,7.5,110.4,NaN,138.7,NaN,...,315.6,285.3,1.3,357.3,254.5,NaN,527.6,47.9,218.0,4867.4


In [310]:
combined_df.drop([4960,4961], inplace=True)

In [311]:
combined_df.to_csv('/Users/aryankhurana/Electricity-Forecasting-Indian-States/Electricity_Consumption_Dataset.csv', index=False)

In [296]:
elec_data = pd.read_csv('/Users/aryankhurana/Electricity-Forecasting-Indian-States/Electricity_Consumption_Dataset.csv')
elec_data.head(50)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,DNH,DVC,...,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption,Pondy
0,2013-01-06,251.5,1.6,21.3,40.9,3.8,60.3,5.8,15.0,56.2,...,192.0,1.7,254.5,127.6,2.6,225.0,33.6,107.5,NaN,5.6
1,2013-01-07,257.9,1.7,21.2,39.6,3.8,64.3,5.8,15.1,59.9,...,192.9,1.8,262.3,129.2,2.6,219.1,34.2,106.6,NaN,5.5
2,2013-01-08,265.1,1.6,20.4,39.2,3.9,63.0,5.9,15.2,60.3,...,197.1,1.8,256.6,131.9,2.1,238.7,33.9,103.9,NaN,5.8
3,2013-01-09,261.0,1.6,21.4,39.5,3.7,65.0,5.8,15.2,60.0,...,197.6,1.9,254.6,135.0,2.7,229.1,32.7,105.7,NaN,5.9
4,2013-01-10,260.0,1.6,21.1,39.7,3.8,66.5,5.9,14.9,58.9,...,199.9,1.9,255.3,130.0,3.2,224.5,35.4,105.3,NaN,5.9
5,2013-01-11,254.5,1.5,20.9,34.3,3.6,64.3,5.8,14.7,57.5,...,197.5,2.0,256.3,132.0,2.7,222.9,33.4,107.4,NaN,5.9
6,2013-01-12,257.4,1.6,20.2,39.9,3.4,65.1,5.8,14.8,55.6,...,195.4,1.5,248.8,130.9,3.0,222.6,32.4,97.8,NaN,5.8
7,2013-07-15,221.1,1.2,24.8,45.0,5.9,63.8,6.0,13.3,61.2,...,142.8,1.3,250.4,118.6,3.0,242.5,35.0,135.7,NaN,6.2
8,2013-07-16,221.3,1.1,24.7,43.5,5.7,63.9,6.2,14.4,61.2,...,146.3,1.3,249.2,115.9,3.0,246.6,34.2,135.7,NaN,6.1
9,2013-07-17,211.8,1.3,25.4,43.9,6.0,65.2,5.9,14.2,61.2,...,152.0,1.3,251.6,124.7,3.0,248.8,33.4,135.7,NaN,6.0


In [295]:
elec_data['Total Consumption'] = elec_data.iloc[:, 1:].sum(axis=1)
elec_data.head(50)

,Dates,Andhra Pradesh,Arunachal Pradesh,Assam,Bihar,Chandigarh,Chhattisgarh,DD,DNH,DVC,...,Rajasthan,Sikkim,Tamil Nadu,Telangana,Tripura,UP,Uttarakhand,West Bengal,Total Consumption,Pondy
0,2013-01-06,251.5,1.6,21.3,40.9,3.8,60.3,5.8,15.0,56.2,...,192.0,1.7,254.5,127.6,2.6,225.0,33.6,107.5,8537.1,5.6
1,2013-01-07,257.9,1.7,21.2,39.6,3.8,64.3,5.8,15.1,59.9,...,192.9,1.8,262.3,129.2,2.6,219.1,34.2,106.6,8630.1,5.5
2,2013-01-08,265.1,1.6,20.4,39.2,3.9,63.0,5.9,15.2,60.3,...,197.1,1.8,256.6,131.9,2.1,238.7,33.9,103.9,8769.0,5.8
3,2013-01-09,261.0,1.6,21.4,39.5,3.7,65.0,5.8,15.2,60.0,...,197.6,1.9,254.6,135.0,2.7,229.1,32.7,105.7,8736.3,5.9
4,2013-01-10,260.0,1.6,21.1,39.7,3.8,66.5,5.9,14.9,58.9,...,199.9,1.9,255.3,130.0,3.2,224.5,35.4,105.3,8762.4,5.9
5,2013-01-11,254.5,1.5,20.9,34.3,3.6,64.3,5.8,14.7,57.5,...,197.5,2.0,256.3,132.0,2.7,222.9,33.4,107.4,8648.7,5.9
6,2013-01-12,257.4,1.6,20.2,39.9,3.4,65.1,5.8,14.8,55.6,...,195.4,1.5,248.8,130.9,3.0,222.6,32.4,97.8,8514.9,5.8
7,2013-07-15,221.1,1.2,24.8,45.0,5.9,63.8,6.0,13.3,61.2,...,142.8,1.3,250.4,118.6,3.0,242.5,35.0,135.7,8461.5,6.2
8,2013-07-16,221.3,1.1,24.7,43.5,5.7,63.9,6.2,14.4,61.2,...,146.3,1.3,249.2,115.9,3.0,246.6,34.2,135.7,8494.2,6.1
9,2013-07-17,211.8,1.3,25.4,43.9,6.0,65.2,5.9,14.2,61.2,...,152.0,1.3,251.6,124.7,3.0,248.8,33.4,135.7,8567.1,6.0


In [ ]:
elec_data = elec_data.sort_values('Dates')
elec_data = elec_data.reset_index(drop=True)
elec_data.head(50)
